In [1]:
# load libraries
import os
import glob
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

# Set global float format to 2 decimal places with thousands separator
pd.options.display.float_format = '{:,.7f}'.format
pd.set_option('display.max_columns', None)
np.set_printoptions(suppress = True, precision=4)

# ---------------- storing paths of all input files -------------
input_path = '../data'
all_files = []
for dirname, _, filenames in os.walk(input_path):
    for filename in filenames:
        all_files.append(os.path.join(dirname, filename))

# ------------ training dates ---------------
train_dates = [
    '2026-01-01_2026-01-10', 
    '2026-01-11_2026-01-20', 
    '2026-02-01_2026-02-10', 
    '2026-02-11_2026-02-20',
    '2026-03-01_2026-03-11',
]

test_dates = [
    '2026-01-21_2026-01-31', 
    '2026-02-21_2026-02-28', 
    '2026-03-12_2026-03-20', 
]

train_files = [f for f in all_files if any(d in f for d in train_dates)]
test_files = [f for f in all_files if any(d in f for d in test_dates)]

In [2]:
# load the ground truth fuel consumption data for jan, feb, mar

s1 = pd.read_csv("../data/smry_jan_train_ordered.csv")
s2 = pd.read_csv("../data/smry_feb_train_ordered.csv")
s3 = pd.read_csv("../data/smry_mar_train_ordered.csv")

In [3]:
# clean raw telemetry - parse timestamps, remove duplicates, sort by vehicle and time

def preprocess_telemetry(df):
    """
    Apply pre processing filters.
    """
    df = df[df['vehicle'].str.startswith("Dump", na=False)].copy()
    df['ts'] = pd.to_datetime(df['ts'])
    df = df[df['satellites'] >= 7]
    df = df[df['gnss_pdop'] <= 3].copy()
    df['working_date'] = (df['ts'] + pd.Timedelta(hours=2)).dt.date
    df.loc[df['speed'] > 2.0, 'ignition'] = 1
    df_clean = df.dropna(subset=['latitude', 'longitude'])

    # No need to convert latitude and longitude since I didn't use maps for spatial analysis
    
    # gdf = gpd.GeoDataFrame(
    #     df_clean, 
    #     geometry=gpd.points_from_xy(df_clean.longitude, df_clean.latitude),
    #     crs="EPSG:4326"
    # )
    
    # gdf = gdf.to_crs(epsg=32645)
    # gdf['x_m'] = gdf.geometry.x
    # gdf['y_m'] = gdf.geometry.y
    
    return df_clean

In [4]:
# load raw telemetry parquet files

df0 = pd.read_parquet(train_files[0])
df1 = pd.read_parquet(train_files[1])
df2 = pd.read_parquet(train_files[2])
df3 = pd.read_parquet(train_files[3])
df4 = pd.read_parquet(train_files[4])

In [5]:
# apply preprocessing to each telemetry chunk

df0 = preprocess_telemetry(df0)
df1 = preprocess_telemetry(df1)
df2 = preprocess_telemetry(df2)
df3 = preprocess_telemetry(df3)
df4 = preprocess_telemetry(df4)

In [6]:
# combine all telemetry chunks into a list for iteration

telemetry_dfs = [df0, df1, df2, df3, df4]

In [7]:
# load and combine all summary csv files (ground truth fuel labels)

def load_smry_data(data_dir='../data'):
    smry_files = glob.glob(f'{data_dir}/smry_*_train_ordered.csv')
    df_list = []
    
    for f in smry_files:
        df = pd.read_csv(f)
        df_list.append(df)
        
    master_smry = pd.concat(df_list, ignore_index=True)
    master_smry['date'] = pd.to_datetime(master_smry['date']).dt.date
    return master_smry

smry_df = load_smry_data()

In [8]:
# Feature engineering and moving from timestamps to shift aggregated data.
# aggregate raw telemetry rows into one row per (vehicle, date, shift)
# extracts ignition hours, moving hours, idle hours, distance, altitude gain, uphill speed etc

def extract_shift_features(df_chunk):
    df_chunk = df_chunk.sort_values(['vehicle', 'ts'])
    df_chunk['duration_hours'] = df_chunk.groupby('vehicle')['ts'].diff().dt.total_seconds() / 3600.0
    df_chunk['duration_hours'] = df_chunk['duration_hours'].fillna(0)
    df_chunk['duration_hours'] = df_chunk['duration_hours'].clip(upper=2)

    shift_records = []
    grouped = df_chunk.groupby(['vehicle', 'working_date', 'shift_dpr'], observed=True)

    for (veh, date, shift), group in grouped:
        if group.empty:
            continue

        # Time features
        is_ignition = (group['ignition'] == 1)
        is_off = (group['ignition'] == 0)
        is_moving = is_ignition & (group['speed'] >= 1.0)
        is_idle = is_ignition & (group['speed'] < 1.0)

        ignition_hours = group.loc[is_ignition, 'duration_hours'].sum()
        moving_hours = group.loc[is_moving, 'duration_hours'].sum()
        idle_hours = group.loc[is_idle, 'duration_hours'].sum()
        off_hours = group.loc[is_off, 'duration_hours'].sum()

        total_dist = group['disthav'].sum() / 1000.0

        alt_diff = group['altitude'].diff()
        cum_altitude_gain = alt_diff[alt_diff > 0].sum()
        cum_altitude_loss = alt_diff[alt_diff < 0].abs().sum()

        is_climbing = (alt_diff > 0.5) & (group['speed'] > 1.0)
        uphill_speed_mean = group.loc[is_climbing, 'speed'].mean()
        if pd.isna(uphill_speed_mean):
            uphill_speed_mean = 0.0

            
        uphill_hrs = group.loc[is_climbing, 'duration_hours'].sum()
        
        is_flat = (alt_diff >= -0.5) & (alt_diff <= 0.5) & (group['speed'] > 1.0)
        flat_hrs = group.loc[is_flat, 'duration_hours'].sum()
        
        is_downhill = (alt_diff < -0.5) & (group['speed'] > 1.0)
        downhill_hrs = group.loc[is_downhill, 'duration_hours'].sum()

        gradient = ((alt_diff[alt_diff > 0]) / (group['disthav'] + 0.01)).sum()
        stop_go_intensity = ((group['speed'] > 1.0) & (group['speed'].shift(-1) <= 1.0)).sum()

        shift_records.append({
            'vehicle': veh,
            'working_date': date,
            'shift': shift,
            
            'ignition_hours': ignition_hours,
            'moving_hours': moving_hours,
            'uphill_hrs' : uphill_hrs,
            'flat_hrs' : flat_hrs,
            'downhill_hrs' : downhill_hrs,
            'idle_hours': idle_hours,
            'off_hours': off_hours,
            
            'total_dist': total_dist,
            'cum_altitude_gain': cum_altitude_gain,
            'cum_altitude_loss': cum_altitude_loss,
            
            'gradients': gradient,
            'stop_go': stop_go_intensity,
            'speed_sum' : group['speed'].sum(),
            'ax_sum' : group['axis_x'].sum(),
            
            'ratio': moving_hours / (ignition_hours + 1e-6),
            'ratio*dist' : moving_hours / (ignition_hours + 1e-6) * total_dist,
        })

    return pd.DataFrame(shift_records)



In [9]:
# run feature extraction on each telemetry chunk

all_features = []

for i, t_df in enumerate(telemetry_dfs):
    print(f"Extracting features from chunk {i}...")
    
    t_df = t_df.sort_values(by=['vehicle', 'ts'])
    chunk_features = extract_shift_features(t_df)
    all_features.append(chunk_features)

Extracting features from chunk 0...
Extracting features from chunk 1...
Extracting features from chunk 2...
Extracting features from chunk 3...
Extracting features from chunk 4...


In [10]:
# merge extracted features with ground truth fuel data on (vehicle, date, shift)

master_features_df = pd.concat(all_features, ignore_index=True)

final_train_df = pd.merge(
    master_features_df, 
    smry_df, 
    left_on=['vehicle', 'working_date', 'shift'],
    right_on=['vehicle', 'date', 'shift'], 
    how='inner'  
)

In [11]:
# remove rows with physically impossible data
# eg trucks reporting >7 L/km, or burning fuel with engine off
    
def bad_data_filter(final_train_df):
    df_clean = final_train_df.copy()
    df_clean['fuel_efficiency'] = df_clean['acons'] / (df_clean['total_dist'] + 0.01)
    
    mask = (
        (df_clean['fuel_efficiency'] > 7.5) | 
        ((df_clean['fuel_efficiency'] < 2.5) & (df_clean['total_dist'] > 10.0)) |
        ((df_clean['ignition_hours'] < 1.0) & (df_clean['acons'] > 100)) |
        ((df_clean['total_dist'] > 50) & (df_clean['fuel_efficiency'] < 1.5)) |
        ((df_clean['vehicle'] == 'Dump031') & (df_clean['fuel_efficiency'] > 5.5))
    )
    
    bad_data_mask = (
        ((df_clean['cum_altitude_loss'] < 500) & (df_clean['acons'] > 100)) |
        ((df_clean['ignition_hours'] > 0.4) & (df_clean['acons'] == 0)) |
        ((df_clean['ignition_hours'] < 0.05) & (df_clean['acons'] != 0)) |
        mask
    )
    
    deleted_rows = df_clean[bad_data_mask]
    df_clean = df_clean[~bad_data_mask].copy()
    
    df_clean = df_clean.drop(columns=['fuel_efficiency'])
    
    print(f"removed {len(deleted_rows)} rows")
    return df_clean

In [12]:
# re-merge and apply the bad data filter to get clean training data

final_train_df = pd.merge(
    master_features_df, 
    smry_df, 
    left_on=['vehicle', 'working_date', 'shift'],
    right_on=['vehicle', 'date', 'shift'], 
    how='inner'
)

final_train_clean = bad_data_filter(final_train_df)
print(final_train_clean.shape)

removed 597 rows
(3865, 27)


In [22]:
# compute each trucks historical average fuel efficiency (L/km) and add as a feature
# this gives the model a baseline for how efficient each truck is

vehicle_totals = final_train_clean.groupby('vehicle', observed=True)[['acons', 'total_dist']].sum().reset_index()
vehicle_totals['historical_L_per_km'] = vehicle_totals['acons'] / (vehicle_totals['total_dist'] + 0.1)

efficiency_dict = vehicle_totals.set_index('vehicle')['historical_L_per_km'].to_dict()
final_train_clean['historical_L_per_km'] = final_train_clean['vehicle'].map(efficiency_dict)

global_efficiency = final_train_clean['acons'].sum() / (final_train_clean['total_dist'].sum() + 0.1)
final_train_clean['historical_L_per_km'] = final_train_clean['historical_L_per_km'].fillna(global_efficiency)

print("Top 5 inefficient trucks:")
print(vehicle_totals.sort_values(by='historical_L_per_km', ascending=False)[['vehicle', 'historical_L_per_km']].head(5))


Top 5 inefficient trucks:
    vehicle  historical_L_per_km
15  Dump029            5.0288628
16  Dump030            4.9063734
23  Dump037            4.8470485
29  Dump043            4.8394330
17  Dump031            4.8380690


In [23]:
# define thresholds for the two-stage prediction
# shifts with near-zero ignition and distance are directly predicted as 0
# only shifts above MODEL_THRESHOLD fuel are used for model training

MODEL_THRESHOLD       = 2.0
ZERO_RULE_IGNITION_H  = 0.05
ZERO_RULE_DIST_KM     = 0.1

def is_zero_shift(df):
    is_dead = (df['ignition_hours'] < 0.1) & (df['total_dist'] < 0.1) 
    is_missing = df['ignition_hours'].isna()
    
    return is_dead | is_missing

def split_train_by_stage(df_clean):
    zero_mask = is_zero_shift(df_clean) | (df_clean['acons'] < MODEL_THRESHOLD)
    df_model  = df_clean[~zero_mask].copy()
    df_zero   = df_clean[zero_mask].copy()
    print(f"Model training rows  : {len(df_model):,}  "
          f"(acons {df_model['acons'].min():.1f} {df_model['acons'].max():.1f})")
    print(f"rows skipped: {len(df_zero):,}")
    return df_model, df_zero

In [24]:
# prepare features and target for modeling
# drop columns that leak info or arent useful (dates, raw refill data etc)
# convert vehicle and shift to categorical type

def prepare_data_for_modeling(df_clean):
    df = df_clean.copy()
    y = df['acons'] 
    
    cols_to_drop = [
        'working_date', 'date', 'mine',
        'initlev', 'endlev', 'arefill', 'runhrs', 'lph', 
        'acons'
    ]
    
    existing_drops = [c for c in cols_to_drop if c in df.columns]
    X = df.drop(columns=existing_drops)
    
    if 'vehicle' in X.columns:
        X['vehicle'] = X['vehicle'].astype('category')
        
    if 'shift' in X.columns:
        X['shift'] = X['shift'].astype('category')
        
    return X, y

In [25]:
# train catboost regressor with categorical support for vehicle and shift
# uses early stopping on validation set to prevent overfitting

import numpy as np
import pandas as pd

def train_catboost(X_train, y_train, X_test, y_test):
    print("CatBoost :")
    
    cat_features = ['vehicle', 'shift']
    
    model_cb = CatBoostRegressor(
        iterations=3000,             
        learning_rate=0.015,         
        depth=10,                    
        colsample_bylevel=0.8,       
        cat_features=cat_features,   
        early_stopping_rounds=100,   
        l2_leaf_reg=5.0,        
        min_data_in_leaf=20,    
        thread_count=-1           
    )

    model_cb.fit(
        X_train, y_train,
        eval_set=(X_test, y_test),   
        verbose=100
    )
    
    preds_cb = np.maximum(model_cb.predict(X_test), 0)
    zero_ignition_mask = (X_test['ignition_hours'] == 0)
    preds_cb[zero_ignition_mask] = 0.0
    
    mse_cb = mean_squared_error(y_test, preds_cb)
    
    print(f"CatBoost mse: {mse_cb}")
    
    # Display Feature Importances
    importances = model_cb.get_feature_importance()
    importance_df = pd.DataFrame({
        'Feature': X_train.columns,
        'Importance': importances
    }).sort_values(by='Importance', ascending=False)
    
    print("\nCatBoost Feature Importances : ")
    print(importance_df)
    
    return model_cb, preds_cb

# df_model is used for training, df_zero are rows directly predicted to 0
df_model, df_zero = split_train_by_stage(final_train_clean)

# Time based splitting
df_model = df_model.sort_values(by='working_date').reset_index(drop=True)
split_idx = int(len(df_model) * 0.8)

df_train_chunk = df_model.iloc[:split_idx].copy()
df_test_chunk  = df_model.iloc[split_idx:].copy()

X_train, y_train = prepare_data_for_modeling(df_train_chunk)
X_test,  y_test  = prepare_data_for_modeling(df_test_chunk)

print(f"Temporal Split : Train: {len(X_train)} rows | Validation: {len(X_test)} rows")

# Random split
# X, y = prepare_data_for_modeling(df_model)
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

cat_model, cat_preds = train_catboost(X_train, y_train, X_test, y_test)

Model training rows  : 3,216  (acons 2.2 375.7)
rows skipped: 649
Temporal Split : Train: 2572 rows | Validation: 644 rows
CatBoost :
0:	learn: 75.7218981	test: 82.4692934	best: 82.4692934 (0)	total: 8.54ms	remaining: 25.6s
100:	learn: 27.7297397	test: 31.6655083	best: 31.6655083 (100)	total: 591ms	remaining: 17s
200:	learn: 18.1653269	test: 22.1947710	best: 22.1947710 (200)	total: 1.28s	remaining: 17.8s
300:	learn: 15.6982148	test: 20.6762629	best: 20.6762629 (300)	total: 2.02s	remaining: 18.1s
400:	learn: 14.5252152	test: 20.3378886	best: 20.3378886 (400)	total: 2.74s	remaining: 17.8s
500:	learn: 13.6333941	test: 20.2492513	best: 20.2479304 (496)	total: 3.5s	remaining: 17.4s
600:	learn: 12.8943410	test: 20.1765112	best: 20.1765112 (600)	total: 4.28s	remaining: 17.1s
700:	learn: 12.2465137	test: 20.1548527	best: 20.1492305 (695)	total: 5.07s	remaining: 16.6s
800:	learn: 11.6781318	test: 20.1337689	best: 20.1272360 (788)	total: 5.86s	remaining: 16.1s
900:	learn: 11.1691800	test: 20.115

In [26]:
id_mapping = pd.read_csv('../data/id_mapping_new.csv')
id_mapping['date'] = pd.to_datetime(id_mapping['date']).dt.date

In [18]:
# load and preprocess test telemetry files
    
test_dfs = []
for f in test_files:
    print(f"Loading {f}")
    df = pd.read_parquet(f)
    df = preprocess_telemetry(df)
    test_dfs.append(df)
    print(f"{len(df)} rows after preprocessing")

Loading ../data/telemetry_2026-01-21_2026-01-31.parquet
3575816 rows after preprocessing
Loading ../data/telemetry_2026-02-21_2026-02-28.parquet
3006583 rows after preprocessing
Loading ../data/telemetry_2026-03-12_2026-03-20.parquet
3057240 rows after preprocessing


In [19]:
# extract shift features from test telemetry using the same function as training

all_test_features = []
for i, t_df in enumerate(test_dfs):
    print(f"extract test features from chunk {i}...")
    chunk_features = extract_shift_features(t_df)
    all_test_features.append(chunk_features)
    print(f"{len(chunk_features)} shift records")

test_features_df = pd.concat(all_test_features, ignore_index=True)
print(f"\nTotal test shift records extracted: {len(test_features_df)}")

extract test features from chunk 0...
940 shift records
extract test features from chunk 1...
708 shift records
extract test features from chunk 2...
783 shift records

Total test shift records extracted: 2431


In [27]:
# two stage prediction:
# stage 1 - if ignition and distance are near zero, predict 0 directly
# stage 2 - for actual working shifts, use the trained model

def predict_two_stage(df_submit, cat_model, X_train):
    """
    Stage 1: directly predict 0 for some rows
    Stage 2: use catboost for other rows
    """
    predictions = np.zeros(len(df_submit))

    zero_mask  = is_zero_shift(df_submit)
    model_mask = ~zero_mask

    if model_mask.sum() > 0:
        X_m = df_submit[model_mask][X_train.columns].copy()
        X_m['vehicle'] = pd.Categorical(
            X_m['vehicle'], categories=X_train['vehicle'].cat.categories
        )
        X_m['shift'] = pd.Categorical(
            X_m['shift'], categories=X_train['shift'].cat.categories
        )
        
        preds = np.maximum(cat_model.predict(X_m), 0)

        predictions[model_mask.values] = preds

    print(f"Zero rows : {zero_mask.sum():,}")
    print(f"Model rows : {model_mask.sum():,}")
    return predictions

In [28]:
# merge test features with id mapping, run predictions, apply post-processing fixes
# save final submission csv

merged = pd.merge(
    id_mapping,
    test_features_df,
    left_on=['vehicle', 'date', 'shift'],
    right_on=['vehicle', 'working_date', 'shift'],
    how='left'
)

merged['historical_L_per_km'] = merged['vehicle'].map(efficiency_dict)
merged['historical_L_per_km'] = merged['historical_L_per_km'].fillna(global_efficiency)

matched = merged['ignition_hours'].notna().sum()
missing = merged['ignition_hours'].isna().sum()
print(f"Matched {matched}/{len(id_mapping)} rows")
if missing > 0:
    print(f"{missing} id_mapping rows have no telemetry")

train_feature_cols = list(X_train.columns)
X_submit = merged[train_feature_cols].copy()

numeric_cols = X_submit.select_dtypes(include=[np.number]).columns
X_submit[numeric_cols] = X_submit[numeric_cols].fillna(0)

# Match categorical levels to training set 
X_submit['vehicle'] = pd.Categorical(X_submit['vehicle'], categories=X_train['vehicle'].cat.categories)
X_submit['shift'] = pd.Categorical(X_submit['shift'], categories=X_train['shift'].cat.categories)

print(f"X_submit shape: {X_submit.shape}")
print(f"Unseen vehicles (will be NaN): {X_submit['vehicle'].isna().sum()}")

predictions = predict_two_stage(X_submit, cat_model, X_train)
n_zeros = (predictions == 0).sum()

# Build submission
submission = pd.DataFrame({
    'id': merged['id'].astype(int),
    'fuel_consumption': predictions
})
submission = submission.sort_values('id').reset_index(drop=True)

submission.to_csv('../submissions/submission1.csv', index=False)
print(f"\nSaved to ../submissions/submission.csv")

print(submission.head(5))

Matched 1701/1735 rows
34 id_mapping rows have no telemetry
X_submit shape: (1735, 19)
Unseen vehicles (will be NaN): 6
Zero rows : 354
Model rows : 1,381

Saved to ../submissions/submission.csv
   id  fuel_consumption
0   0         0.0000000
1   1       153.2643987
2   2        73.9124694
3   3         7.3649281
4   4        81.3245995
